# CMS 기능군 분석 데이터 품질 검토

## tl;dr

- 최신 CMS 캐시 기준으로 기능군 판매 분석은 구현 가능하다. 판매 매출의 상품 마스터 연결률은 **99.56%**, 보정 후 기능구분 미분류 매출 비중은 **0%**다.
- 매출 상위 10개 기능구분 조합이 분석 대상 매출의 **85.37%**를 차지한다. 상위 58개 SKU는 **69.98%**를 차지하며 17개 기능구분 조합에 걸쳐 있다.
- 재고·운송 데이터의 기능구분 SKU 연결률은 99%대지만, 미입고(open PO)는 **80.54%**로 낮다. 다만 미입고 수량 기준 연결률은 **92.92%**다.
- CMS의 기능구분 1·2를 그대로 필터로 쓰면 `크림`처럼 상위 분류에 따라 뜻이 달라진다. 화면용 `분석기능군` 매핑을 별도로 두는 것이 안전하다.

## Context & Methods

### Key Assumptions

- 판매 분석 범위는 2024-07-15~2026-07-14의 EU 현지 판매 캐시다.
- 기존 애플리케이션과 동일하게 EU 판매 Biz Type 필터, 기본/사용자 기능구분 보정, 비핵심 상품군 제외를 적용한다.
- 재고·운송·미입고 연결성은 2026-07-14에 생성된 최신 발주 검토 캐시를 사용한다.
- 기능군 후보 순위는 판매금액(EUR) 기준이며, 수요 지표는 별도로 판매수량을 유지한다.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'backend' / 'storage' / 'cms_fetch_cache').is_dir():
            return candidate
    raise FileNotFoundError('ESM_SCM8 project root not found')

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))
SEASON_CACHE = ROOT / 'backend/storage/cms_fetch_cache/506889b9d7106139006ce741b6a26082134d6e322661526969a867213e639b36.json'
ORDER_CACHE = ROOT / 'backend/storage/cms_fetch_cache/30d6d305b90084a91df9ce9ef53ca175fe5dba8fc6779878b7c1801c66c5d4e2.json'

season_raw = json.loads(SEASON_CACHE.read_text(encoding='utf-8'))['raw']
order_raw = json.loads(ORDER_CACHE.read_text(encoding='utf-8'))['raw']
products = pd.DataFrame(season_raw['prod_list'])
sales = pd.DataFrame(season_raw['sales_history'])

def clean_key(series: pd.Series) -> pd.Series:
    return series.fillna('').astype(str).str.strip()

def blank_mask(series: pd.Series) -> pd.Series:
    return series.isna() | series.astype(str).str.strip().str.casefold().isin({'', 'nan', 'none', '<na>', '미분류'})

def pct(numerator: float, denominator: float) -> float:
    return round(100 * numerator / denominator, 2) if denominator else 0.0

print({'project_root': ROOT.name, 'season_cache_mb': round(SEASON_CACHE.stat().st_size / 1e6, 1), 'order_cache_mb': round(ORDER_CACHE.stat().st_size / 1e6, 1)})

## Data

### 1. 상품 마스터와 판매 원천 품질

In [ ]:
product_codes = clean_key(products['prod_cd'])
sales_codes = clean_key(sales['prod_cd'])
sales_qty = pd.to_numeric(sales['qty'], errors='coerce')
sales_amount = pd.to_numeric(sales['amount'], errors='coerce')
matched = sales_codes.isin(set(product_codes))
dates = pd.to_datetime(sales['ship_dt'], errors='coerce')

product_profile = pd.DataFrame([
    {'check': '상품 마스터 행', 'value': len(products), 'rate_pct': 100.0},
    {'check': '고유 상품코드', 'value': product_codes.nunique(), 'rate_pct': pct(product_codes.nunique(), len(products))},
    {'check': '기능구분1 누락', 'value': int(blank_mask(products['class1_nm']).sum()), 'rate_pct': pct(blank_mask(products['class1_nm']).sum(), len(products))},
    {'check': '기능구분2 누락', 'value': int(blank_mask(products['class2_nm']).sum()), 'rate_pct': pct(blank_mask(products['class2_nm']).sum(), len(products))},
])
sales_join_profile = pd.DataFrame([
    {'check': '판매 행 연결', 'coverage_pct': pct(matched.sum(), len(sales))},
    {'check': '판매수량 연결', 'coverage_pct': pct(sales_qty[matched].sum(), sales_qty.sum())},
    {'check': '판매금액 연결', 'coverage_pct': pct(sales_amount[matched].sum(), sales_amount.sum())},
])
display(product_profile)
display(sales_join_profile)
print({'sales_rows': len(sales), 'sales_unique_sku': sales_codes.nunique(), 'unmatched_unique_sku': sales_codes[~matched].nunique(), 'date_from': str(dates.min().date()), 'date_to': str(dates.max().date()), 'negative_qty_rows': int((sales_qty < 0).sum()), 'negative_amount_rows': int((sales_amount < 0).sum())})

## Results

### 2. 기존 분석 로직 적용 후 기능구분 커버리지와 집중도

In [ ]:
from backend.analysis import merge_default_category_corrections, apply_category_corrections_to_merged
from core.season_calendar import (
    merge_sales_with_product_master, exclude_season_category1_values,
    PRODUCT_CODE_COL, CATEGORY1_COL, CATEGORY2_COL, QTY_COL, AMOUNT_COL,
    is_official_category1, is_official_category2, _standard_prod_df,
)

corrections = merge_default_category_corrections(pd.DataFrame())
merged = merge_sales_with_product_master(sales, products, eu_local=True)
merged = apply_category_corrections_to_merged(merged, corrections, PRODUCT_CODE_COL, CATEGORY1_COL, CATEGORY2_COL)
merged = exclude_season_category1_values(merged)

sku_sales = (
    merged.groupby([PRODUCT_CODE_COL, CATEGORY1_COL, CATEGORY2_COL], dropna=False)
    .agg(qty=(QTY_COL, 'sum'), amount=(AMOUNT_COL, 'sum'))
    .reset_index().sort_values('amount', ascending=False)
)
category_sales = (
    sku_sales.groupby([CATEGORY1_COL, CATEGORY2_COL], dropna=False)
    .agg(amount=('amount', 'sum'), qty=('qty', 'sum'), sku_count=(PRODUCT_CODE_COL, 'nunique'))
    .reset_index().sort_values('amount', ascending=False).reset_index(drop=True)
)
category_sales['amount_rank'] = category_sales.index + 1
category_sales['amount_share_pct'] = (category_sales['amount'] / category_sales['amount'].sum() * 100).round(2)
category_sales['qty_rank'] = category_sales['qty'].rank(method='min', ascending=False).astype(int)

bad_category2 = ~pd.Series([is_official_category2(a, b) for a, b in zip(merged[CATEGORY1_COL], merged[CATEGORY2_COL], strict=False)], index=merged.index)
concentration = pd.DataFrame([
    {'top_sku_count': n, 'amount_share_pct': pct(sku_sales.head(n)['amount'].sum(), sku_sales['amount'].sum()), 'function_pairs': sku_sales.head(n)[[CATEGORY1_COL, CATEGORY2_COL]].drop_duplicates().shape[0]}
    for n in (10, 20, 58, 100)
])
display(concentration)
display(category_sales.head(10)[[CATEGORY1_COL, CATEGORY2_COL, 'sku_count', 'qty', 'amount', 'amount_share_pct', 'amount_rank', 'qty_rank']])
print({'effective_sales_rows': len(merged), 'effective_unique_sku': merged[PRODUCT_CODE_COL].nunique(), 'category1_unmapped_sku': merged.loc[blank_mask(merged[CATEGORY1_COL]), PRODUCT_CODE_COL].nunique(), 'category2_unmapped_sku': merged.loc[blank_mask(merged[CATEGORY2_COL]), PRODUCT_CODE_COL].nunique(), 'top10_category_share_pct': round(category_sales.head(10)['amount_share_pct'].sum(), 2), 'nonstandard_category2_sku': merged.loc[bad_category2, PRODUCT_CODE_COL].nunique(), 'nonstandard_category2_amount_pct': pct(merged.loc[bad_category2, AMOUNT_COL].sum(), merged[AMOUNT_COL].sum())})

### 3. 재고·판매·운송·미입고의 기능구분 연결성

In [ ]:
product_dimension = _standard_prod_df(products)
product_dimension = apply_category_corrections_to_merged(product_dimension, corrections, PRODUCT_CODE_COL, CATEGORY1_COL, CATEGORY2_COL)
mapped_codes = set(product_dimension[PRODUCT_CODE_COL].astype(str))

operational_rows = []
for dataset_name in ['stock_local', 'stock_hq', 'sales_local', 'sales_hq', 'shipping', 'open_po']:
    frame = pd.DataFrame(order_raw[dataset_name])
    codes = clean_key(frame['prod_cd'])
    valid = codes.ne('')
    mapped = valid & codes.isin(mapped_codes)
    measure_field = next((name for name in ['sales_qty_3m', 'avbl_qty', 'qty', 'open_qty'] if name in frame.columns), None)
    measure = pd.to_numeric(frame[measure_field], errors='coerce').fillna(0) if measure_field else pd.Series(0, index=frame.index)
    operational_rows.append({
        'dataset': dataset_name,
        'rows': len(frame),
        'unique_sku': codes[valid].nunique(),
        'mapped_unique_sku': codes[mapped].nunique(),
        'sku_coverage_pct': pct(codes[mapped].nunique(), codes[valid].nunique()),
        'measure': measure_field,
        'measure_coverage_pct': pct(measure[mapped].abs().sum(), measure[valid].abs().sum()) if measure[valid].abs().sum() else None,
    })
operational_coverage = pd.DataFrame(operational_rows)
display(operational_coverage)

## Takeaways

1. **판매 기반 기능군 분석은 바로 구현 가능하다.** 기존 시즌 분석의 SKU-상품마스터 결합과 기능구분 보정 로직을 재사용하면 된다.
2. **화면용 기능군은 파생 차원으로 관리해야 한다.** `썬케어 > 크림`과 `스킨케어 > 크림`을 같은 이름으로 노출하면 오해가 생긴다.
3. **미입고 연결률 개선이 우선이다.** 판매·재고·운송은 충분한 반면 open PO의 SKU 연결률은 80.54%다. 전체 상품 마스터를 추가 조회하거나 미판매 신규 SKU를 보충해야 한다.
4. **기능군 상위 10개를 우선 노출하는 전략은 데이터로 지지된다.** 상위 10개 기능구분 조합이 매출의 85.37%를 포괄한다.
5. **API 계약을 보강해야 한다.** 저장된 OpenAPI 명세에는 `/eu/products`가 없고 실제 응답 필드와 차이가 있으므로, 구현 전 명세 갱신과 스키마 회귀 테스트가 필요하다.